# Treinar o Fast-SCNN do zero

Exemplo de como treinar o Fast-SCNN com imagens e máscaras. Pastas `data/train/img` e `data/train/label` para os dados de treino.

In [ ]:
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('Nome da GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Check min/max values in training masks
import os
import numpy as np
from PIL import Image
mask_dir = r'd:\Documentos\Git\edge-segmentation-lab\data\train\label'
mask_files = [f for f in os.listdir(mask_dir) if f.lower().endswith('.png')]
max_val = -1
min_val = 9999
for fname in mask_files[:100]:
    mask = np.array(Image.open(os.path.join(mask_dir, fname)).convert('L'))
    max_val = max(max_val, mask.max())
    min_val = min(min_val, mask.min())
print(f'Min value in masks: {min_val}')
print(f'Max value in masks: {max_val}')

In [ ]:
# carrega máscaras de IDs Cityscapes e faz one-hot em tempo real
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Mapeamento cor RGB -> trainId (0-18, 255=ignore)
# Paleta oficial de cores Cityscapes para visualização
cityscapes_color_to_train = {
    (128, 64, 128): 0,    # road (roxo)
    (244, 35, 232): 1,    # sidewalk (rosa)
    (70, 70, 70): 2,      # building (cinza escuro)
    (102, 102, 156): 3,   # wall (cinza azulado)
    (190, 153, 153): 4,   # fence (bege)
    (153, 153, 153): 5,   # pole (cinza)
    (250, 170, 30): 6,    # traffic light (amarelo)
    (220, 220, 0): 7,     # traffic sign (amarelo brilhante)
    (107, 142, 35): 8,    # vegetation (verde oliva)
    (152, 251, 152): 9,   # terrain (verde claro)
    (70, 130, 180): 10,   # sky (azul)
    (220, 20, 60): 11,    # person (vermelho)
    (255, 0, 0): 12,      # rider (vermelho brilhante)
    (0, 0, 142): 13,      # car (azul escuro)
    (0, 0, 70): 14,       # truck (azul muito escuro)
    (0, 60, 100): 15,     # bus (azul)
    (0, 80, 100): 16,     # train (azul esverdeado)
    (0, 0, 230): 17,      # motorcycle (azul)
    (119, 11, 32): 18,    # bicycle (marrom)
}

def rgb_to_trainid(mask_rgb):
    '''Converte máscara RGB para trainIds usando aproximação de cor mais próxima.'''
    h, w = mask_rgb.shape[:2]
    trainid_mask = 255 * np.ones((h, w), dtype=np.uint8)
    
    # Criar array de cores da paleta
    palette_colors = np.array(list(cityscapes_color_to_train.keys()))
    palette_trainids = np.array(list(cityscapes_color_to_train.values()))
    
    # Para cada pixel, encontrar cor mais próxima na paleta
    mask_flat = mask_rgb.reshape(-1, 3).astype(np.int32)
    
    for i in range(len(mask_flat)):
        pixel = mask_flat[i]
        # Distância euclidiana para cada cor da paleta
        distances = np.sqrt(np.sum((palette_colors - pixel)**2, axis=1))
        closest_idx = np.argmin(distances)
        
        # Se a distância for muito grande (>50), considerar como ignore
        if distances[closest_idx] < 50:
            trainid_mask.flat[i] = palette_trainids[closest_idx]
    
    return trainid_mask

def trainid_to_onehot(trainid_img, num_classes=19):
    '''Converte máscara trainId para one-hot (H, W, num_classes).'''
    h, w = trainid_img.shape
    onehot = np.zeros((h, w, num_classes), dtype=np.float32)
    for c in range(num_classes):
        onehot[:, :, c] = (trainid_img == c).astype(np.float32)
    return onehot

class CityscapesDataset(Dataset):
    '''Dataset Cityscapes: carrega imagens e máscaras de IDs, converte para one-hot em tempo real.'''
    def __init__(self, img_dir, mask_dir, img_height=96, img_width=256, num_classes=19):
        self.img_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.png')])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_height = img_height
        self.img_width = img_width
        self.num_classes = num_classes
        
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        # Carregar imagem
        img = Image.open(os.path.join(self.img_dir, self.img_files[idx])).convert('RGB')
        img = img.resize((self.img_width, self.img_height))
        img = np.array(img, dtype=np.float32) / 255.0  # Normalizar [0, 1]
        
        # Carregar máscara RGB e converter para trainIds
        mask = Image.open(os.path.join(self.mask_dir, self.mask_files[idx]))
        mask = mask.resize((self.img_width, self.img_height), Image.NEAREST)
        mask = np.array(mask, dtype=np.uint8)
        
        # Converter RGB -> trainId -> one-hot
        trainid_mask = rgb_to_trainid(mask)
        mask_onehot = trainid_to_onehot(trainid_mask, self.num_classes)
        
        # Converter para tensores PyTorch: (C, H, W) para imagem, (H, W, C) para máscara
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask_onehot = torch.tensor(mask_onehot).float()
        
        return img, mask_onehot

# Exemplo de uso:
img_dir = r'd:\Documentos\Git\edge-segmentation-lab\data\train\img'
mask_dir = r'd:\Documentos\Git\edge-segmentation-lab\data\train\label'
dataset = CityscapesDataset(img_dir, mask_dir)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f'Dataset criado: {len(dataset)} imagens')
print(f'Num classes: 19 (trainIds Cityscapes)')

In [ ]:
# Teste: carregar e visualizar uma imagem do dataset
import matplotlib.pyplot as plt

# Testar com a imagem train23.png
test_idx = dataset.img_files.index('train23.png')
img_tensor, mask_onehot_tensor = dataset[test_idx]

# Converter tensor de volta para numpy para visualizar
img_np = img_tensor.permute(1, 2, 0).numpy()
mask_onehot_np = mask_onehot_tensor.numpy()

print(f'Shape da imagem: {img_np.shape}')  # (H, W, 3)
print(f'Shape da máscara one-hot: {mask_onehot_np.shape}')  # (H, W, num_classes)
print(f'Valor mínimo imagem: {img_np.min()}, máximo: {img_np.max()}')
print(f'Valor mínimo máscara: {mask_onehot_np.min()}, máximo: {mask_onehot_np.max()}')
print(f'Classes presentes na máscara: {np.where(mask_onehot_np.sum(axis=(0,1)) > 0)[0]}')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_np)
axes[0].set_title('Imagem Original')
axes[0].axis('off')

# Converter one-hot de volta para índices de classe para visualizar
mask_class_idx = np.argmax(mask_onehot_np, axis=-1)
axes[1].imshow(mask_class_idx, cmap='tab20')
axes[1].set_title('Máscara (classes)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Check unique colors in mask
from PIL import Image
import numpy as np

mask_path = r'd:\Documentos\Git\edge-segmentation-lab\data\train\label\train23.png'
mask_rgb = np.array(Image.open(mask_path).convert('RGB'))

unique_colors = np.unique(mask_rgb.reshape(-1, 3), axis=0)
print(f'Total unique colors in mask: {len(unique_colors)}')
print('\nFirst 20 colors found:')
for i, color in enumerate(unique_colors[:20]):
    print(f'{i}: {tuple(color)}')

In [ ]:
# Analyze most common RGB colors in mask
mask_path = r'd:\Documentos\Git\edge-segmentation-lab\data\train\label\train23.png'
mask = np.array(Image.open(mask_path))

from collections import Counter
pixels_flat = mask.reshape(-1, 3)
pixel_tuples = [tuple(p) for p in pixels_flat]
color_counts = Counter(pixel_tuples)

print('Top 30 most frequent RGB colors:')
for i, (color, count) in enumerate(color_counts.most_common(30)):
    pct = 100 * count / len(pixel_tuples)
    print(f'{i:2d}. RGB{color} - {count:5d} pixels ({pct:5.2f}%)')
    
print(f'\nTotal unique colors: {len(color_counts)}')